###  Goal

- The goal is to ask a LLM to convert long-form transcript into course notes. I also want it to provide a mermaid diagram.
- I'm trying to decide whether chunking is neccessary. 
    - gpt-5.5 and 5.4 has the following context limits:
        - window: 1,050,000
        - max output tokens: 128k
    - gpt 5.4 mini has the following context limits:
        - window: 400k
        - max output tokens: 128k
    - knowledge cutoff
        - 5.5 -> Dec 1, 2025
        - 5.4 and 5.4 mini -> Aug 31, 2025

### Experimentation
- [x] Count the number of tokens per transcript. This will give me insight into how many tokens out of the entire window that each transcript will take it. This will give me an idea as to whether chunking is neccessary.

### Result
- Token count for module 1 is ~ 26k and the context window for gpt-5.4 is >1m, so I can pass the entire module and see what output I get from it. 
- Next step:
    - Pass the entire module to gpt and evaluate the output. 


In [ ]:
# read transcripts in path, 1_ingestion/transcripts_module1

from pathlib import Path

In [8]:
transcript_path = Path("../1_ingestion/transcripts_module1")

# test transcript_path
transcript_path.exists()


True

In [13]:
# list filepaths
filepaths = [f for f in transcript_path.iterdir()]
filepaths

[PosixPath('../1_ingestion/transcripts_module1/transcript_CeukwyUdaz8.json'),
 PosixPath('../1_ingestion/transcripts_module1/transcript_VRrEEVeJ440.json'),
 PosixPath('../1_ingestion/transcripts_module1/transcript_OH_R0Sl9neM.json'),
 PosixPath('../1_ingestion/transcripts_module1/transcript_Qa0-jYtRdbY.json'),
 PosixPath('../1_ingestion/transcripts_module1/transcript_Crm_5n4mvmg.json'),
 PosixPath('../1_ingestion/transcripts_module1/transcript_zZyKUeOR4Gg.json'),
 PosixPath('../1_ingestion/transcripts_module1/transcript_j9kcEuGcC2Y.json'),
 PosixPath('../1_ingestion/transcripts_module1/transcript_pqQFlV3f9Bo.json'),
 PosixPath('../1_ingestion/transcripts_module1/transcript_dCa3JvmJbr0.json'),
 PosixPath('../1_ingestion/transcripts_module1/transcript_0j3XK5PsnxA.json')]

In [22]:
import json

# investigate the content in one transcript

with open(filepaths[0], mode="r", encoding="utf-8") as f:
    test_transcript = json.load(f)

In [53]:
test_transcript.keys()

dict_keys(['id', 'title', 'duration', 'complete_transcript', 'transcript_w_ts'])

In [23]:
type(test_transcript)

dict

In [24]:
test_transcript.keys()

dict_keys(['id', 'title', 'duration', 'complete_transcript', 'transcript_w_ts'])

In [26]:
test_transcript["complete_transcript"]

"welcome to the second lesson of our first session of machinery and zoom comp and we will talk we will compare um machine learning systems with rule based systems in the previous lesson we talked about an example where how we can use machine learning for predicting the price of a car and now in this lesson we will compare um like the old way of doing things without machine learning and with uh using machine learning and we will illustrate this using spam detection example so imagine that you have a system so this is a system for emails right so usually you have gmail or outlook somewhere you have emails and you use this for talking to your colleagues for work related things for chatting with friends and so on so like each row here is an email right and everything works well so you can send emails receive emails and then at some point users of our system start complaining about unsolicited emails so it can be promotions about discounts that people have aren't subscribed to so this unsol

### Notes about JSON
- A JSON file is stored on disk as plain text, so Python reads it as a str, with `f.read()`
- To turn that JSON text into Python data, use `json.loads()`
- Use `json.load(f)` for a file object

In [1]:
import tiktoken

In [37]:
# count tokens for test_transcript

model_name = "gpt-5"
encoding = tiktoken.encoding_for_model(model_name=model_name)
len(encoding.encode(test_transcript["complete_transcript"]))


2622

In [38]:
# does gpt-4 tokenizer provide the same token count? nope.
model_name = "gpt-4"
encoding = tiktoken.encoding_for_model(model_name=model_name)
len(encoding.encode(test_transcript["complete_transcript"]))


2669

In [34]:
print(tiktoken.__version__)
print(tiktoken.list_encoding_names())

0.13.0
['gpt2', 'r50k_base', 'p50k_base', 'p50k_edit', 'cl100k_base', 'o200k_base', 'o200k_harmony']


In [35]:
import tiktoken.model

print(tiktoken.model.MODEL_TO_ENCODING)

{'o1': 'o200k_base', 'o3': 'o200k_base', 'o4-mini': 'o200k_base', 'gpt-5': 'o200k_base', 'gpt-4.1': 'o200k_base', 'gpt-4o': 'o200k_base', 'gpt-4': 'cl100k_base', 'gpt-3.5-turbo': 'cl100k_base', 'gpt-3.5': 'cl100k_base', 'gpt-35-turbo': 'cl100k_base', 'davinci-002': 'cl100k_base', 'babbage-002': 'cl100k_base', 'text-embedding-ada-002': 'cl100k_base', 'text-embedding-3-small': 'cl100k_base', 'text-embedding-3-large': 'cl100k_base', 'text-davinci-003': 'p50k_base', 'text-davinci-002': 'p50k_base', 'text-davinci-001': 'r50k_base', 'text-curie-001': 'r50k_base', 'text-babbage-001': 'r50k_base', 'text-ada-001': 'r50k_base', 'davinci': 'r50k_base', 'curie': 'r50k_base', 'babbage': 'r50k_base', 'ada': 'r50k_base', 'code-davinci-002': 'p50k_base', 'code-davinci-001': 'p50k_base', 'code-cushman-002': 'p50k_base', 'code-cushman-001': 'p50k_base', 'davinci-codex': 'p50k_base', 'cushman-codex': 'p50k_base', 'text-davinci-edit-001': 'p50k_edit', 'code-davinci-edit-001': 'p50k_edit', 'text-similarity

### Notes on tiktoken

To count tokens:

- use `tiktoken.encoding_for_model()` if you want to insert a model name. 
- it could sometimes be useful to check the mapping between encoding name and model name

    ```
    import tiktoken.model

    print(tiktoken.model.MODEL_TO_ENCODING) 
    ```
- token count vary depending on the tokenizer used by different model types; not all models use the same tokenzier and that's why the count is different depending on the llm used
- the mapping in tiktoken provides the accurate count of tokens for only the models that are present in the mapping. For example, tiktoken provides access to the tokenizer of the gpt-5 model but not gpt5.4 so using the tokenizer for gpt-5 gives the estimate for the token count of the document rather than the accurate token count

In [44]:
from os import PathLike

In [ ]:
def count_tokens(document: str, model_name: str="gpt-5", ) -> int:

    """Estimates token count based on GPT-5 tokenizer"""

    encoding = tiktoken.encoding_for_model(model_name=model_name)
    token_count = len(encoding.encode(document))
    return token_count

def read_json(filepath:str| PathLike) -> str:

    """Reads a file path and returns video id and associating transcript """

    path = Path(filepath)
    
    with open(path, mode="r", encoding="utf-8") as f:
        doc_dict = json.load(f)
        video_id = doc_dict["id"]
        complete_transcript = doc_dict["complete_transcript"]
        
    return video_id, complete_transcript

In [57]:
read_json(filepaths[0])[1]

"welcome to the second lesson of our first session of machinery and zoom comp and we will talk we will compare um machine learning systems with rule based systems in the previous lesson we talked about an example where how we can use machine learning for predicting the price of a car and now in this lesson we will compare um like the old way of doing things without machine learning and with uh using machine learning and we will illustrate this using spam detection example so imagine that you have a system so this is a system for emails right so usually you have gmail or outlook somewhere you have emails and you use this for talking to your colleagues for work related things for chatting with friends and so on so like each row here is an email right and everything works well so you can send emails receive emails and then at some point users of our system start complaining about unsolicited emails so it can be promotions about discounts that people have aren't subscribed to so this unsol

In [52]:
print(filepaths)

[PosixPath('../1_ingestion/transcripts_module1/transcript_CeukwyUdaz8.json'), PosixPath('../1_ingestion/transcripts_module1/transcript_VRrEEVeJ440.json'), PosixPath('../1_ingestion/transcripts_module1/transcript_OH_R0Sl9neM.json'), PosixPath('../1_ingestion/transcripts_module1/transcript_Qa0-jYtRdbY.json'), PosixPath('../1_ingestion/transcripts_module1/transcript_Crm_5n4mvmg.json'), PosixPath('../1_ingestion/transcripts_module1/transcript_zZyKUeOR4Gg.json'), PosixPath('../1_ingestion/transcripts_module1/transcript_j9kcEuGcC2Y.json'), PosixPath('../1_ingestion/transcripts_module1/transcript_pqQFlV3f9Bo.json'), PosixPath('../1_ingestion/transcripts_module1/transcript_dCa3JvmJbr0.json'), PosixPath('../1_ingestion/transcripts_module1/transcript_0j3XK5PsnxA.json')]


In [63]:
token_count_module1 = []

for i in filepaths:
    transcript = read_json(i)
    video_id = transcript[0]
    complete_transcript = transcript[1]

    token_count = count_tokens(complete_transcript)
    word_count = len(complete_transcript.split())

    token_count_module1.append({
        "video_id": video_id,
        "token_count": token_count,
        "word_count": word_count
    })
    

In [65]:
import pandas as pd

df = pd.DataFrame(token_count_module1)
df 


,video_id,token_count,word_count
0,CeukwyUdaz8,2622,2575
1,VRrEEVeJ440,1007,998
2,OH_R0Sl9neM,3003,2936
3,Qa0-jYtRdbY,3058,2973
4,Crm_5n4mvmg,1502,1494
5,zZyKUeOR4Gg,3556,3480
6,j9kcEuGcC2Y,2827,2778
7,pqQFlV3f9Bo,1163,1133
8,dCa3JvmJbr0,3249,3220
9,0j3XK5PsnxA,4179,4122


In [66]:
df["token_count"].sum()

np.int64(26166)

- is <class 'pathlib._local.PosixPath'> a string or is it a distinct type
    - distinct type -> test by `isinstance(path, str)`
    - type-hint it as <Path>
    - make it more flexible by importing <PathLike> from os -> `from os import PathLike`
- token count vs word count
    - token count is useful to guage how many tokens will be used when interacting with llms
    - word count is useful for human intuition -- it helps us guage the length of the document